# Import helper functions

In [ ]:
%run ../0.shared_notebooks/0_helper_functions.ipynb

# Set constants

In [ ]:
CASE="DIVD-2025-00018"
SUB="stealers"
IN_DIR="../IN"
OUT_DIR="../NORMALIZED"
ERR_DIR="../ERR"

In [ ]:
!ls $IN_DIR
!ls $ERR_DIR

## Defaults


In [ ]:
defaults = [
  "",  # ts_found (Timestamp when the data was "found")
  "",  # ts_leaked (Timestamp when the data was stolen/leaked)
  "0",     # has_name (0/1 if the record has a name)
  "0",     # has_dob (0/1 if the record has a date of birth)
  "0",     # has_addr (0/1 if the record has a address)
  "0",     # has_phone (0/1 if the record has a )
  "0",     # has_cc (0/1 if the record has creditcard data)
  "0",     # has bankacc (0/1 if the record has a bank account)
  "0",     # has_ssn (0/1 if the record has a ssn)
  "0",     # has ip (0/1 if the record has an ip address)
  "0",     # extra_data (json object with extra data)
]
ts_from_file = False

## Read in files, guess column and type

In [ ]:
files=sorted(glob(f"{IN_DIR}/*.duckdb"))

In [ ]:
files

In [ ]:
goodlines = 0
badlines = 0
for file in files:
    filename = os.path.basename(file)
    outfile = open(f"{OUT_DIR}/{filename.replace('.duckdb','.tsv')}","w")
    outfile.write("\t".join(["username","passwd","url", "ts_found", "ts_leaked", "has_name", "has_dob", "has_addr", "has_phone", "has_cc", "has_bankacc", "has_ssn", 
                             "has_ip", "extra_data" ]))
    outfile.write("\n")
    src_duck = duckdb.connect(file)
    cursor = src_duck.execute("select username, masked_password, url, ts_found from entity")
    row = cursor.fetchone()
    while row is not None:
        extra_data = ""
        fields = [ str(row[0]), str(row[1]), str(row[2]) ]
        fields.extend(defaults)
        fields[3] = str(row[3])
        fields[-1] = json.dumps(extra_data)
        fields2 = [f.replace('\t', '\\t') for f in fields]
        fields = fields2
        if len(fields) != 14 or "\t" in "".join(fields) :
            print(f"\n{fields}")
            bla()
        outfile.write("\t".join(fields))
        outfile.write("\n")
        row = cursor.fetchone()
        goodlines = goodlines + 1
        if goodlines % 1000 == 0:
            print(f"{goodlines:,}", end="\r")

print(f"{goodlines:,}", end="\r")

In [ ]:
goodlines

In [ ]:
!ls $OUT_DIR
!ls -l $ERR_DIR